# ChEBI BioT5 Diverse Beam Collection On Kaggle

This notebook clones the repo, ensures the ChEBI-20 dataset exists locally, runs one partition of the active BioT5 grouped collection pipeline with diverse beam search, and exports a clean artifact bundle for downstream merge and training notebooks.

Each part run also writes a downloadable zip that contains the full stage folder `collect_chebi_biot5_part_<n>/`. When you download multiple parts locally, extract all of those zips into the same parent `thesis_artifacts/` folder before re-uploading them for `05_merge_biot5_collection_parts.ipynb`.


In [ ]:
from pathlib import Path
import sys

REPO_URL = "https://github.com/mruniverse8/Thesis.git"
REPO_BRANCH = "gflownet"
REPO_DIR = Path("/kaggle/working/Thesis")

%cd /kaggle/working
!if [ -d "{REPO_DIR / '.git'}" ]; then echo "Reusing {REPO_DIR}"; elif [ -d "{REPO_DIR}" ]; then echo "Existing non-git directory at {REPO_DIR}; delete it and rerun the notebook." && false; else git clone --depth 1 "{REPO_URL}" "{REPO_DIR}"; fi
!git -C "{REPO_DIR}" fetch --depth 1 origin "{REPO_BRANCH}" && git -C "{REPO_DIR}" checkout -B "{REPO_BRANCH}" FETCH_HEAD

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from thesis_kaggle_support import (
    ARTIFACT_EXPORT_ROOT,
    create_zip_archive,
    dump_yaml,
    ensure_paths_exist,
    ensure_runtime_dependencies,
    export_stage_artifacts,
    json_dumps,
    load_yaml,
    read_json,
    report_runtime,
)

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

huggingface_login_status = "not_attempted"
access_token = None
try:
    access_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    huggingface_login_status = f"secret_unavailable:{exc.__class__.__name__}"

if access_token:
    login(token=access_token)
    huggingface_login_status = "logged_in"
elif huggingface_login_status == "not_attempted":
    huggingface_login_status = "no_huggingface_token"


In [ ]:
NUM_PARTS = 5
PART_INDEX = 2
TARGET_MOLECULES_PER_DESCRIPTION = 100
MAX_DESCRIPTIONS = None

if PART_INDEX < 1 or PART_INDEX > NUM_PARTS:
    raise ValueError(f"PART_INDEX must be between 1 and {NUM_PARTS}, got {PART_INDEX}")

PART_TAG = f"part_{PART_INDEX}_of_{NUM_PARTS}"
STAGE_NAME = f"collect_chebi_biot5_part_{PART_INDEX}"
CHEBI_OUTPUT_DIR = REPO_DIR / "data" / "chebi20"
COLLECTION_OUTPUT_DIR = REPO_DIR / "data_collection" / "outputs" / f"kaggle_chebi20_biot5_train_{PART_TAG}"
DERIVED_TRAIN_FILE = REPO_DIR / "data" / "post_training" / "processed" / f"train_multimol_{PART_TAG}.jsonl"
TEMP_CONFIG_PATH = REPO_DIR / "kaggle" / "generated_configs" / f"collect_biot5_chebi20.{PART_TAG}.kaggle.yaml"
ARTIFACT_ZIP_PATH = ARTIFACT_EXPORT_ROOT / f"{STAGE_NAME}.zip"

ensure_runtime_dependencies(REPO_DIR)
runtime_report = report_runtime(require_gpu=True)
print(json_dumps({
    "runtime": runtime_report,
    "huggingface_login_status": huggingface_login_status,
    "stage_name": STAGE_NAME,
    "num_parts": NUM_PARTS,
    "part_index": PART_INDEX,
    "target_molecules_per_description": TARGET_MOLECULES_PER_DESCRIPTION,
    "max_descriptions": MAX_DESCRIPTIONS,
    "collection_output_dir": str(COLLECTION_OUTPUT_DIR),
    "derived_train_file": str(DERIVED_TRAIN_FILE),
    "artifact_zip_path": str(ARTIFACT_ZIP_PATH),
}))


In [ ]:
CHEBI_PROCESSED_DIR = CHEBI_OUTPUT_DIR / "processed"
have_processed = all((CHEBI_PROCESSED_DIR / f"{split}.jsonl").exists() for split in ("train", "validation", "test"))
print(json_dumps({
    "have_processed": have_processed,
    "processed_dir": str(CHEBI_PROCESSED_DIR),
}))

%cd {REPO_DIR}
!if [ -f "{CHEBI_PROCESSED_DIR / 'train.jsonl'}" ] && [ -f "{CHEBI_PROCESSED_DIR / 'validation.jsonl'}" ] && [ -f "{CHEBI_PROCESSED_DIR / 'test.jsonl'}" ]; then echo "ChEBI processed splits already exist"; else python scripts/download_chebi20.py --output-dir "{CHEBI_OUTPUT_DIR}"; fi

config = load_yaml(REPO_DIR / "configs" / "collect_biot5_chebi20.yaml")
config["model"]["model_max_length"] = int(
    config["model"].get("model_max_length", config["generation"].get("max_length", 512))
)
config["data"]["train_file"] = str(CHEBI_PROCESSED_DIR / "train.jsonl")
config["data"]["staging_dir"] = str(COLLECTION_OUTPUT_DIR)
config["data"]["derived_train_file"] = str(DERIVED_TRAIN_FILE)
target_count = int(TARGET_MOLECULES_PER_DESCRIPTION)
config["generation"]["target_molecules_per_description"] = target_count
config["generation"]["num_return_sequences"] = target_count
num_beam_groups = int(config["generation"].get("num_beam_groups", 1))
num_beams = max(int(config["generation"].get("num_beams", target_count)), target_count)
if num_beam_groups > 1:
    num_beams = ((num_beams + num_beam_groups - 1) // num_beam_groups) * num_beam_groups
config["generation"]["num_beams"] = num_beams
if num_beam_groups > 1 and config["generation"]["num_beams"] % num_beam_groups != 0:
    raise ValueError(
        "Kaggle collection config requires num_beams to be divisible by num_beam_groups; "
        f"got {config['generation']['num_beams']} and {num_beam_groups}."
    )
config.setdefault("runtime", {})["description_offset"] = 0
config["runtime"]["max_descriptions"] = None if MAX_DESCRIPTIONS is None else int(MAX_DESCRIPTIONS)
config["runtime"]["num_parts"] = int(NUM_PARTS)
config["runtime"]["part_index"] = int(PART_INDEX)
dump_yaml(config, TEMP_CONFIG_PATH)
print(f"Wrote config: {TEMP_CONFIG_PATH}")
print(TEMP_CONFIG_PATH.read_text(encoding="utf-8"))


In [ ]:
%cd {REPO_DIR}
!python scripts/collect_biot5_chebi20.py --config "{TEMP_CONFIG_PATH}" --num-parts {NUM_PARTS} --part-index {PART_INDEX}


In [ ]:
required_outputs = ensure_paths_exist({
    "chebi_processed_dir": CHEBI_PROCESSED_DIR,
    "collection_output_dir": COLLECTION_OUTPUT_DIR,
    "collection_summary": COLLECTION_OUTPUT_DIR / "summary.json",
    "derived_train_file": DERIVED_TRAIN_FILE,
})
artifact_dir, manifest = export_stage_artifacts(
    stage_name=STAGE_NAME,
    artifact_map={
        "chebi20_processed": CHEBI_PROCESSED_DIR,
        "collection_outputs": COLLECTION_OUTPUT_DIR,
        "post_training_processed/train_multimol.jsonl": DERIVED_TRAIN_FILE,
    },
    metadata={
        "required_outputs": required_outputs,
        "config_path": str(TEMP_CONFIG_PATH),
        "part_index": PART_INDEX,
        "num_parts": NUM_PARTS,
        "summary": read_json(COLLECTION_OUTPUT_DIR / "summary.json"),
    },
)
artifact_zip_path = create_zip_archive(artifact_dir, ARTIFACT_ZIP_PATH)
print(json_dumps({
    "artifact_dir": str(artifact_dir),
    "artifact_zip_path": str(artifact_zip_path),
    "local_extract_parent": "thesis_artifacts",
    "local_extract_stage_dir": f"thesis_artifacts/{STAGE_NAME}",
    "manifest": manifest,
}))
